
# PPO pairwise LA patch notebook

This notebook does **targeted pairwise repair**.

Instead of broad routed distillation, it patches one checkpoint using:

- a **student checkpoint** that has a known weakness
- a **peer teacher checkpoint** that is better on that weakness
- optional **adjudicator checkpoints**
- optional **lookahead judge**
- a **focus bank** from the weak matchup, for example `student vs LA-2`
- a **preserve bank** from matchups that should not get worse

Typical jobs:

- `PPO_827 <- patch from PPO_832 on LA-2`
- `PPO_832 <- patch from PPO_827 on LA-3`

The notebook is configurable and can run either one selected job or all configured jobs.


In [1]:

import copy
import json
import time
import math
import pickle
import random
import re
from collections import OrderedDict, Counter
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

from C4.CNet192 import save_cnet192
from C4.connect4_env import Connect4Env
from C4.fast_connect4_lookahead import Connect4Lookahead
from PPO.actor_critic import ActorCritic, TransferCfg, NEG_INF

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda



## Config

Edit `PATCH_JOBS` first.

Each job defines:

- `student_init`: checkpoint to patch
- `peer_teacher`: checkpoint that should teach the weak matchup
- `focus_depth`: the `LA-x` weakness to repair
- `adjudicator_models`: optional checkpoint judges
- `la_judge_depth`: optional lookahead judge
- `preserve_depths`: lookaheads to preserve while patching
- `output_tag`: checkpoint name for the patched result

The defaults below implement the two pairwise ideas we discussed.


In [2]:

# ============================================================
# Global config
# ============================================================

SEED = 666
MODEL_DETERMINISTIC = True
PATCH_SUFFIX = int(time.time())

RUN_ALL_JOBS = False
SELECT_JOB_INDEX = 0

OUTPUT_DIR = Path("PAIRWISE_PATCH")
DATA_DIR = OUTPUT_DIR / "data"
MODEL_DIR = Path("PPO_Models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

SAVE_PATCH_DATASET = True
SAVE_RESULTS_XLSX = True
SAVE_MODEL_CHECKPOINT = True
SAVE_JSON_SUMMARY = True

# ============================================================
# Eval cache
# ============================================================

USE_EVAL_CACHE = True
SAVE_EVAL_CACHE = True
EVAL_CACHE_PATH = OUTPUT_DIR / "pairwise_eval_cache.json"

# Optional external cache files to merge in once, if they exist.
# Example:
#   CACHE_BOOTSTRAP_PATHS = [Path("soup_fast_eval_cache.json")]
CACHE_BOOTSTRAP_PATHS = []

# ============================================================
# Collection config
# ============================================================

MAX_FOCUS_POSITIONS = 6000
MAX_PRESERVE_POSITIONS = 2400

# Focus rows are usually the scarce signal. Preserve rows can be deduplicated harder.
DEDUP_FOCUS_STATES = False
DEDUP_PRESERVE_STATES = False

MIN_FOCUS_PLY = 5
MAX_FOCUS_PLY = 24
MIN_PRESERVE_PLY = 4
MAX_PRESERVE_PLY = 28

# Keep mostly games where the student does not win.
FOCUS_KEEP_ONLY_NONWINS = True
PRESERVE_KEEP_ONLY_NONWINS = False

# ------------------------------------------------------------
# Opening control
#
# "empty_only"   -> always start from empty board
# "random_only"  -> always add random opening plies
# "mixed"        -> sometimes empty, sometimes random
# "forced_only"  -> only use any explicit forced sequence passed in
#
# For your current comparison against GS on empty board,
# empty_only is the clean default.
# ------------------------------------------------------------
OPENING_MODE = "empty_only"
EXTRA_RANDOM_OPENING_PLIES_CHOICES = [0, 1, 2]
OPENING_RANDOM_PROB = 0.55


# ============================================================
# Evaluation mode and primary metric
#
# This notebook uses EMPTY-BOARD evaluation only.
# The primary headline metric is GLOBAL_SCORE, matching the
# project GS weighting on empty-board starts (base=1.4).
# GS_OPEN is intentionally not used here.
# ============================================================

EVAL_MODE_TAG = "trusted_empty_board_gs_v2"
GLOBAL_SCORE_WEIGHT_BASE = 1.4
PRIMARY_EVAL_METRIC = "GLOBAL_SCORE"

# ============================================================
# Target construction config
# ============================================================

SOFT_TARGET_POWER = 1.0

# Focus rows: how much to trust peer teacher / adjudicators / LA judge
PEER_WEIGHT = 1.00
ADJUDICATOR_MODEL_WEIGHT = 0.70
LA_JUDGE_WEIGHT = 1.10

# When teacher and judge disagree, blend toward the judge
LA_JUDGE_BLEND = 0.65

# If True, on focus rows where LA judge disagrees with peer teacher,
# use the judge more aggressively for the hard target / soft blend.
USE_LA_JUDGE_HARD_TARGET_ON_FOCUS = True

# Require some support before overriding anchor behavior on focus rows.
REQUIRE_TEACHER_OR_ADJUDICATOR_SUPPORT = True
MIN_SUPPORT_COUNT_FOR_OVERRIDE = 2

# Preserve rows
PRESERVE_TARGET_FROM_ANCHOR = True

# Sample weights
FOCUS_WEIGHT = 1.35
PRESERVE_WEIGHT = 1.5

# ============================================================
# Training config
# ============================================================

BATCH_SIZE = 256
LR = 1.5e-4
WEIGHT_DECAY = 0.0
EPOCHS = 8
EARLY_STOPPING_PATIENCE = 4
MAX_GRAD_NORM = 1.0

LOSS_W_SOFT = 0.65
LOSS_W_HARD = 0.35
LOSS_W_VALUE = 0.0

# Keep the patched model close to the original student.
ANCHOR_LOSS_WEIGHT = 0.60
ANCHOR_FOCUS_MULT = 1.00
ANCHOR_PRESERVE_MULT = 2.30

KEEP_BEST_BY = "mini_eval_PATCH_SCORE"

# ============================================================
# Split config
# ============================================================

VALID_FRAC = 0.18
MIN_VALID_GROUPS = 4
MIN_FOCUS_VALID_GROUPS = 1
MIN_PRESERVE_VALID_GROUPS = 1

# ============================================================
# Evaluation suites
# ============================================================

DISTILL_EVAL_OPPONENTS = OrderedDict({
    "Random":   {"type": "random",    "games": 80},
    "Leftmost": {"type": "leftmost",  "games": 30},
    "Center":   {"type": "center",    "games": 60},
    "LA-1":     {"type": "lookahead", "depth": 1,  "games": 20},
    "LA-2":     {"type": "lookahead", "depth": 2,  "games": 20},
    "LA-3":     {"type": "lookahead", "depth": 3,  "games": 20},
    "LA-4":     {"type": "lookahead", "depth": 4,  "games": 12},
    "LA-5":     {"type": "lookahead", "depth": 5,  "games": 12},
    "LA-6":     {"type": "lookahead", "depth": 6,  "games": 10},
    "LA-7":     {"type": "lookahead", "depth": 7,  "games": 8},
    "LA-9":     {"type": "lookahead", "depth": 9,  "games": 8},
    "LA-11":    {"type": "lookahead", "depth": 11, "games": 6},
    "LA-13":    {"type": "lookahead", "depth": 13, "games": 6},
})

MINI_EVAL_OPPONENTS = OrderedDict({
    "LA-2":   {"type": "lookahead", "depth": 2,  "games": 16},
    "LA-3":   {"type": "lookahead", "depth": 3,  "games": 14},
    "LA-4":   {"type": "lookahead", "depth": 4,  "games": 12},
    "LA-5":   {"type": "lookahead", "depth": 5,  "games": 12},
    "LA-6":   {"type": "lookahead", "depth": 6,  "games": 10},
    "LA-9":   {"type": "lookahead", "depth": 9,  "games": 8},
    "LA-11":  {"type": "lookahead", "depth": 11, "games": 6},
    "LA-13":  {"type": "lookahead", "depth": 13, "games": 6},
})

# ============================================================
# Optional checkpoint prior scores
# Only used as soft relative trust among teachers/adjudicators
# ============================================================

HOF_METASCORES = {
    "PPO_Models/PPO_915.pt": 0.819,
    "PPO_Models/PPO_914.pt": 0.789,
    "PPO_Models/PPO_909.pt": 0.789,
    "PPO_Models/PPO_842.pt": 0.655,
    "PPO_Models/PPO_848.pt": 0.627,
    "PPO_Models/PPO_916.pt": 0.593,
    "PPO_Models/PPO_845.pt": 0.527,
    "PPO_Models/PPO_850.pt": 0.512,
    "PPO_Models/PPO_S2.pt": 0.501,
    "PPO_Models/PPO_917.pt": 0.477,
    "PPO_Models/PPO_849.pt": 0.470,
    "PPO_Models/PPO_812.pt": 0.415,
    "PPO_Models/X_1.pt": 0.385,
    "PPO_Models/PPO_827.pt": 0.355,
    "PPO_Models/PPO_832.pt": 0.352,
    "PPO_Models/PPO_S1.pt": 0.340,
    "PPO_Models/PPO_820.pt": 0.240,
    "PPO_Models/PPO_817.pt": 0.156,
}

# ============================================================
# Patch jobs
# ============================================================

PATCH_JOBS = [
    {
        "tag": "PPO_827_PATCH_FROM_832_LA2",
        "student_init": "PPO_Models/PPO_827.pt",
        "peer_teacher": "PPO_Models/PPO_832.pt",
        "adjudicator_models": ["PPO_Models/X_1.pt", "PPO_Models/PPO_849.pt", "PPO_Models/PPO_850.pt"],
        "la_judge_depth": 4,
        "focus_depth": 2,
        "focus_games_student_vs_la": 250,
        "focus_games_teacher_vs_la": 150,
        "preserve_depths": [3, 4, 5, 6, 7, 9, 11, 13],
        "preserve_games_each": {
            3: 200,
            4: 200,
            5: 200,
            6: 100,
            7: 50,
            9: 30,
            11: 22,
            13: 18,
        },
        "output_tag": f"PPO_827_P{PATCH_SUFFIX}",
    }
]



## Helpers and loading

This section provides:

- environment helpers
- model loading
- opponent logic
- lookahead judge helpers
- evaluation helpers


In [3]:

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

CENTER_ORDER = [3, 4, 2, 5, 1, 6, 0]

def pretty_name(path: str | Path) -> str:
    return Path(path).stem

def center_tiebreak(indices: List[int]) -> int:
    idx_set = set(int(i) for i in indices)
    for c in CENTER_ORDER:
        if c in idx_set:
            return c
    return int(sorted(indices)[0])

def make_env() -> Connect4Env:
    return Connect4Env()

def ensure_state_tensor(state: np.ndarray, device: torch.device) -> torch.Tensor:
    x = torch.as_tensor(state, dtype=torch.float32, device=device)
    if x.dim() == 2:
        x = x.unsqueeze(0).unsqueeze(0)
    elif x.dim() == 3:
        x = x.unsqueeze(0)
    elif x.dim() != 4:
        raise ValueError(f"Unexpected state shape: {tuple(x.shape)}")
    return x

def canonical_state_array(state: np.ndarray) -> np.ndarray:
    s = np.asarray(state)
    if s.ndim == 3:
        return np.ascontiguousarray(s)
    elif s.ndim == 4 and s.shape[0] == 1:
        return np.ascontiguousarray(s[0])
    raise ValueError(f"Unexpected state shape for canonicalization: {s.shape}")

def state_key(state: np.ndarray) -> Tuple[Tuple[int, ...], bytes]:
    s = canonical_state_array(state)
    key_arr = np.ascontiguousarray(np.rint(s).astype(np.int8, copy=False))
    return tuple(key_arr.shape), key_arr.tobytes()

def state_to_board_pov(state: np.ndarray) -> np.ndarray:
    s = np.asarray(state)
    if s.ndim == 4:
        if s.shape[0] != 1:
            raise ValueError(f"Expected batch size 1, got {s.shape}")
        s = s[0]
    if s.ndim == 3:
        if s.shape[0] == 1:
            s = s[0]
        else:
            raise ValueError(f"Expected single-channel POV board, got {s.shape}")
    if s.shape != (6, 7):
        raise ValueError(f"Expected board shape (6,7), got {s.shape}")
    return s.astype(np.int8, copy=False)

def mask_logits_numpy(logits: np.ndarray, legal_actions: List[int]) -> np.ndarray:
    out = np.array(logits, dtype=np.float64, copy=True)
    legal_mask = np.zeros(7, dtype=bool)
    legal_mask[legal_actions] = True
    out[~legal_mask] = -1e18
    return out

def softmax_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = x - np.max(x)
    ex = np.exp(x)
    denom = np.sum(ex)
    if denom <= 0:
        return np.ones_like(x) / len(x)
    return ex / denom

def sharpen_distribution(p: np.ndarray, power: float = 1.0) -> np.ndarray:
    p = np.asarray(p, dtype=np.float64)
    p = np.clip(p, 1e-12, None)
    if power != 1.0:
        p = p ** float(power)
    p = p / p.sum()
    return p

def one_hot_action(action: int, n: int = 7) -> np.ndarray:
    v = np.zeros(n, dtype=np.float32)
    v[int(action)] = 1.0
    return v

def teacher_prior_weight(path_or_name: str) -> float:
    path_or_name = str(path_or_name)
    score = HOF_METASCORES.get(path_or_name, 0.0)
    return float(1.0 + score)


# ============================================================
# Eval cache helpers
# ============================================================

def _normalize_suite_for_cache(suite: Dict[str, Dict]) -> str:
    return json.dumps(list(suite.items()), sort_keys=True)

def make_eval_cache_key(model_path: str | Path, suite: Dict[str, Dict], seed: int, deterministic: bool = True) -> str:
    payload = {
        "deterministic": bool(deterministic),
        "eval_mode_tag": EVAL_MODE_TAG,
        "primary_metric": PRIMARY_EVAL_METRIC,
        "model_path": str(model_path),
        "seed": int(seed),
        "suite": _normalize_suite_for_cache(suite),
    }
    return json.dumps(payload, sort_keys=True)

def load_eval_cache() -> Dict[str, Any]:
    merged = {}

    for p in CACHE_BOOTSTRAP_PATHS:
        p = Path(p)
        if p.exists():
            try:
                merged.update(json.loads(p.read_text(encoding="utf-8")))
                print("Bootstrapped eval cache from:", p)
            except Exception as e:
                print("Could not load bootstrap cache:", p, "|", e)

    if EVAL_CACHE_PATH.exists():
        try:
            merged.update(json.loads(EVAL_CACHE_PATH.read_text(encoding="utf-8")))
            print("Loaded eval cache from:", EVAL_CACHE_PATH)
        except Exception as e:
            print("Could not load eval cache:", EVAL_CACHE_PATH, "|", e)

    return merged

EVAL_CACHE = load_eval_cache()

def save_eval_cache() -> None:
    if not SAVE_EVAL_CACHE:
        return
    EVAL_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    EVAL_CACHE_PATH.write_text(json.dumps(EVAL_CACHE, indent=2), encoding="utf-8")


def load_actor_critic_from_ckpt(path: str | Path, device: torch.device, train: bool = False) -> ActorCritic:
    path = Path(path)
    ac = ActorCritic.from_cnet192_checkpoint(
        path=str(path),
        device=device,
        transfer=TransferCfg(
            strict_load=True,
            freeze_conv=False,
        ),
    )
    ac.train(mode=train)
    return ac

class RandomOpponent:
    def __init__(self, seed: int = 0):
        self.rng = np.random.default_rng(seed)
    def choose(self, state: np.ndarray, legal_actions: List[int]) -> int:
        return int(self.rng.choice(legal_actions))

class LeftmostOpponent:
    def choose(self, state: np.ndarray, legal_actions: List[int]) -> int:
        return int(min(legal_actions))

class CenterOpponent:
    def choose(self, state: np.ndarray, legal_actions: List[int]) -> int:
        return center_tiebreak(list(legal_actions))

class LookaheadOpponent:
    def __init__(self, depth: int):
        self.depth = int(depth)
        self.la = Connect4Lookahead()
        self.la.OPENING_RANDOM = False

    def choose(self, state: np.ndarray, legal_actions: List[int]) -> int:
        if len(legal_actions) == 1:
            return int(legal_actions[0])
        board = state_to_board_pov(state)
        scores = np.asarray(self.la.n_step_action_scores(board, player=1, depth=self.depth), dtype=np.float64)
        mask = np.zeros(7, dtype=bool)
        mask[legal_actions] = True
        scores[~mask] = -1e18
        best = np.max(scores[mask])
        best_cols = [c for c in legal_actions if abs(scores[c] - best) <= 1e-12]
        return center_tiebreak(best_cols)

@torch.no_grad()
def model_action_details(model: ActorCritic, state: np.ndarray, legal_actions: List[int]) -> Dict[str, Any]:
    x = ensure_state_tensor(state, device=device)
    logits_t, value_t = model(x)
    logits = logits_t[0].detach().cpu().numpy().astype(np.float64)
    value = float(value_t[0].detach().cpu().item()) if value_t is not None else 0.0

    masked_logits = mask_logits_numpy(logits, legal_actions)
    probs64 = softmax_np(masked_logits)

    legal_logits = masked_logits[legal_actions]
    best_logit = np.max(legal_logits)
    best_cols = [a for a in legal_actions if abs(float(masked_logits[a]) - float(best_logit)) <= 1e-12]
    top_action = center_tiebreak(best_cols)

    remaining_legal = [a for a in legal_actions if a != top_action]
    if remaining_legal:
        second_logit = max(float(masked_logits[a]) for a in remaining_legal)
        second_best_cols = [a for a in remaining_legal if abs(float(masked_logits[a]) - second_logit) <= 1e-12]
        second_action = center_tiebreak(second_best_cols)
    else:
        second_action = top_action
        second_logit = -1e18

    top_prob = float(probs64[top_action])
    second_prob = float(probs64[second_action]) if second_action != top_action else 0.0
    margin = float(best_logit - second_logit) if second_logit > -1e17 else float(best_logit)

    return {
        "logits": logits,
        "masked_logits": masked_logits,
        "probs": probs64.astype(np.float32),
        "value": value,
        "top_action": int(top_action),
        "second_action": int(second_action),
        "top_prob": top_prob,
        "second_prob": second_prob,
        "margin": margin,
    }

@torch.no_grad()
def choose_model_action(model: ActorCritic, state: np.ndarray, legal_actions: List[int], deterministic: bool = True) -> int:
    details = model_action_details(model, state, legal_actions)
    if deterministic:
        return int(details["top_action"])
    probs = np.asarray(details["probs"], dtype=np.float64)
    probs = probs / probs.sum()
    return int(np.random.choice(np.arange(7), p=probs))

def make_source_from_spec(spec: Dict[str, Any], loaded_models: Dict[str, ActorCritic], seed: int = 0):
    t = spec["type"]
    if t == "model":
        path = spec["path"]
        model = loaded_models[path]
        return {
            "label": pretty_name(path),
            "kind": "model",
            "choose": lambda state, legal, _model=model: choose_model_action(_model, state, legal, deterministic=MODEL_DETERMINISTIC),
        }
    if t == "lookahead":
        opp = LookaheadOpponent(depth=int(spec["depth"]))
        return {
            "label": f"LA-{int(spec['depth'])}",
            "kind": "lookahead",
            "choose": lambda state, legal, _opp=opp: int(_opp.choose(state, legal)),
        }
    if t == "random":
        opp = RandomOpponent(seed=seed)
        return {"label": "Random", "kind": "random", "choose": lambda state, legal, _opp=opp: int(_opp.choose(state, legal))}
    if t == "center":
        opp = CenterOpponent()
        return {"label": "Center", "kind": "center", "choose": lambda state, legal, _opp=opp: int(_opp.choose(state, legal))}
    if t == "leftmost":
        opp = LeftmostOpponent()
        return {"label": "Leftmost", "kind": "leftmost", "choose": lambda state, legal, _opp=opp: int(_opp.choose(state, legal))}
    raise ValueError(f"Unknown source spec: {spec}")

def opponent_weight(label: str, base: float = 1.4) -> float:
    if label in {"Random", "Leftmost", "Center"}:
        return 1.0
    m = re.search(r"(\d+)", label)
    if m is None:
        return 1.0
    depth = int(m.group(1))
    return float(base ** depth)

def summarize_suite_row(row: Dict[str, Any], suite: Dict[str, Dict]) -> Dict[str, Any]:
    score_cols = []
    hard_cols = []
    for label in suite.keys():
        if label in row:
            score_cols.append(label)
            if label in {"LA-2", "LA-3", "LA-4", "LA-5", "LA-6", "LA-9", "LA-11", "LA-13"}:
                hard_cols.append(label)

    avg_score = float(np.mean([row[c] for c in score_cols])) if score_cols else 0.0
    la_hard = float(np.mean([row[c] for c in hard_cols])) if hard_cols else 0.0

    weights = np.array([opponent_weight(c, base=GLOBAL_SCORE_WEIGHT_BASE) for c in score_cols], dtype=np.float64)
    vals = np.array([row[c] for c in score_cols], dtype=np.float64)
    global_score = float((weights * vals).sum() / weights.sum()) if len(score_cols) else 0.0

    row["AVG_SCORE"] = avg_score
    row["LA_HARD"] = la_hard
    row["GLOBAL_SCORE"] = global_score

    # Backward-compatible alias for older cells/files.
    row["GS_CUSTOM"] = global_score
    return row

def play_one_game(model: ActorCritic, opponent_choose, model_starts: bool, seed: int) -> int:
    env = make_env()
    state = env.reset()
    model_turn = bool(model_starts)

    for _ in range(42):
        legal = env.available_actions()
        if not legal:
            return 0
        if model_turn:
            action = choose_model_action(model, state, legal, deterministic=MODEL_DETERMINISTIC)
        else:
            action = int(opponent_choose(state, legal))

        prev_model_turn = model_turn
        state, reward, done = env.step(action)

        if done:
            if reward > 0:
                return +1 if prev_model_turn else -1
            return 0

        model_turn = not model_turn

    return 0

def evaluate_model_on_suite(
    model: ActorCritic,
    suite: Dict[str, Dict],
    loaded_models: Dict[str, ActorCritic],
    model_name: str,
    seed: int = 0,
    show_progress: bool = True,
    tqdm_position: int = 0,
    tqdm_leave: bool = False,
) -> Tuple[pd.DataFrame, Dict]:
    row = OrderedDict()
    row["MODEL"] = model_name
    all_details = {}
    iterable = suite.items()

    if show_progress:
        iterable = tqdm(list(iterable), desc=f"Evaluating {model_name}", leave=tqdm_leave, position=tqdm_position)

    for idx, (label, cfg) in enumerate(iterable):
        games = int(cfg["games"])
        opponent = make_source_from_spec(cfg, loaded_models=loaded_models, seed=seed + 1000 * (idx + 1))
        wins = losses = draws = 0

        for g in range(games):
            model_starts = (g % 2 == 0)
            result = play_one_game(model=model, opponent_choose=opponent["choose"], model_starts=model_starts, seed=seed + idx * 10000 + g)
            if result > 0:
                wins += 1
            elif result < 0:
                losses += 1
            else:
                draws += 1

        score = (wins + 0.5 * draws) / games if games > 0 else 0.0
        row[label] = float(score)
        all_details[label] = {"wins": wins, "losses": losses, "draws": draws, "games": games, "score": float(score)}

        if show_progress:
            iterable.set_postfix({"opp": label, "score": f"{score:.3f}", "W-L-D": f"{wins}-{losses}-{draws}"})

    row = summarize_suite_row(row, suite)
    df = pd.DataFrame([row])
    return df, all_details



def evaluate_checkpoint_path_on_suite_cached(
    model_path: str | Path,
    suite: Dict[str, Dict],
    loaded_models: Dict[str, ActorCritic],
    seed: int = 0,
    show_progress: bool = True,
    tqdm_position: int = 0,
    tqdm_leave: bool = False,
    force_recompute: bool = False,
    verbose: bool = True,
) -> Tuple[pd.DataFrame, Dict]:
    model_path = str(model_path)
    cache_key = make_eval_cache_key(model_path=model_path, suite=suite, seed=seed, deterministic=MODEL_DETERMINISTIC)

    if USE_EVAL_CACHE and (not force_recompute) and cache_key in EVAL_CACHE:
        payload = EVAL_CACHE[cache_key]
        df = pd.DataFrame([payload["row"]])
        details = payload["details"]
        if verbose:
            print(f"[CACHE HIT] {pretty_name(model_path)} | seed={seed}")
        return df, details

    if verbose:
        print(f"[CACHE MISS] {pretty_name(model_path)} | seed={seed} | computing {EVAL_MODE_TAG} eval...")

    model = loaded_models.get(model_path)
    if model is None:
        model = load_actor_critic_from_ckpt(model_path, device=device, train=False)
        model.eval()
        for param in model.parameters():
            param.requires_grad_(False)
        loaded_models[model_path] = model

    df, details = evaluate_model_on_suite(
        model=model,
        suite=suite,
        loaded_models=loaded_models,
        model_name=pretty_name(model_path),
        seed=seed,
        show_progress=show_progress,
        tqdm_position=tqdm_position,
        tqdm_leave=tqdm_leave,
    )

    if USE_EVAL_CACHE:
        EVAL_CACHE[cache_key] = {
            "row": df.iloc[0].to_dict(),
            "details": details,
        }
        save_eval_cache()
        if verbose:
            print(f"[CACHE SAVE] {pretty_name(model_path)}")

    if verbose:
        row = df.iloc[0]
        print(
            f"[DONE] {pretty_name(model_path)} | "
            f"GLOBAL_SCORE={float(row['GLOBAL_SCORE']):.6f} | "
            f"LA_HARD={float(row['LA_HARD']):.6f} | "
            f"AVG_SCORE={float(row['AVG_SCORE']):.6f}"
        )

    return df, details

def display_before_after(pre_eval_df: pd.DataFrame, post_eval_df: pd.DataFrame):
    compare_df = pd.concat(
        [
            pre_eval_df.assign(STAGE="before"),
            post_eval_df.assign(STAGE="after"),
        ],
        ignore_index=True,
    )
    cols = ["STAGE", "MODEL", "LA-2", "LA-3", "LA-4", "LA-5", "LA-6", "LA-9", "LA-11", "LA-13", "AVG_SCORE", "LA_HARD", "GLOBAL_SCORE"]
    cols = [c for c in cols if c in compare_df.columns]
    display(compare_df[cols])


def display_pair_sanity(job: Dict[str, Any], loaded_models: Dict[str, ActorCritic], suite: Dict[str, Dict], seed: int):
    model_paths = [job["student_init"], job["peer_teacher"]] + list(job.get("adjudicator_models", []))
    rows = []

    print("Sanity-check models:")
    for p in model_paths:
        print("  ", pretty_name(p), "->", p)

    outer = tqdm(
        model_paths,
        desc="Pair sanity",
        position=0,
        leave=True,
    )

    for idx, p in enumerate(outer, start=1):
        outer.set_postfix({"model": pretty_name(p)})
        print("" + "-" * 88)
        print(f"Sanity step {idx}/{len(model_paths)}: {pretty_name(p)}")

        df, _ = evaluate_checkpoint_path_on_suite_cached(
            model_path=p,
            suite=suite,
            loaded_models=loaded_models,
            seed=seed,
            show_progress=True,
            tqdm_position=1,
            tqdm_leave=False,
            force_recompute=False,
            verbose=True,
        )

        row = df.iloc[0].to_dict()
        row["MODEL_PATH"] = p
        rows.append(row)

    sanity_df = pd.DataFrame(rows)
    focus_label = f"LA-{int(job['focus_depth'])}"
    cols = ["MODEL", "MODEL_PATH", focus_label, "LA-3", "LA-4", "LA-5", "LA-6", "LA-9", "LA-11", "LA-13", "AVG_SCORE", "LA_HARD", "GLOBAL_SCORE"]
    cols = [c for c in cols if c in sanity_df.columns]

    print("Pair sanity table:")
    display(sanity_df[cols].sort_values(by=[focus_label, "GLOBAL_SCORE"], ascending=False))

def display_eval(df: pd.DataFrame, metric: str = PRIMARY_EVAL_METRIC):
    score_cols = [c for c in df.columns if c.startswith("LA-") or c in {"Random", "Leftmost", "Center"}]
    extra_cols = ["AVG_SCORE", "LA_HARD", "GLOBAL_SCORE"]
    cols = ["MODEL"] + score_cols + [c for c in extra_cols if c in df.columns]
    show = df[cols].copy()
    for c in show.columns:
        if c != "MODEL":
            show[c] = show[c].map(lambda x: f"{float(x):.3f}")
    display(show)
    print(f"Metric [{metric}] =", float(df.iloc[0][metric]))

def compute_patch_score_from_row(row: Dict[str, Any], focus_depth: int) -> float:
    la2  = float(row.get("LA-2", 0.0))
    la3  = float(row.get("LA-3", 0.0))
    la4  = float(row.get("LA-4", 0.0))
    la5  = float(row.get("LA-5", 0.0))
    la6  = float(row.get("LA-6", 0.0))
    la7  = float(row.get("LA-7", 0.0))
    la9  = float(row.get("LA-9", 0.0))
    la11 = float(row.get("LA-11", 0.0))
    la13 = float(row.get("LA-13", 0.0))

    return (
        0.24 * la2  +
        0.18 * la3  +
        0.10 * la4  +
        0.18 * la5  +
        0.06 * la6  +
        0.06 * la7  +
        0.07 * la9  +
        0.05 * la11 +
        0.06 * la13
    )


Loaded eval cache from: PAIRWISE_PATCH\pairwise_eval_cache.json


In [4]:

# ------------------------------------------------------------------
# Trusted empty-board evaluator override
#
# Uses PPO.ppo_agent_eval so GLOBAL_SCORE matches your project eval.
# This replaces the local notebook evaluator for sanity/pre/post/mini eval.
# ------------------------------------------------------------------
from PPO.ppo_agent_eval import (
    evaluate_suite as trusted_evaluate_suite,
    global_score_from_suite_df as trusted_global_score_from_suite_df,
)

_TRUSTED_LOOKAHEAD = Connect4Lookahead()
if hasattr(_TRUSTED_LOOKAHEAD, "OPENING_RANDOM"):
    _TRUSTED_LOOKAHEAD.OPENING_RANDOM = False

def suite_spec_to_eval_cfg(suite: Dict[str, Dict]) -> "OrderedDict[str, int]":
    cfg = OrderedDict()
    for label, spec in suite.items():
        games = int(spec["games"])
        if spec["type"] == "lookahead":
            depth = int(spec["depth"])
            cfg[f"Lookahead-{depth}"] = games
        elif spec["type"] == "random":
            cfg["Random"] = games
        elif spec["type"] == "leftmost":
            cfg["Leftmost"] = games
        elif spec["type"] == "center":
            cfg["Center"] = games
        else:
            raise ValueError(f"Unsupported suite spec for trusted eval: {label} -> {spec}")
    return cfg

def eval_label_to_row_label(label: str) -> str:
    s = str(label)
    if s.startswith("Lookahead-"):
        return "LA-" + s.split("-", 1)[1]
    return s

def evaluate_model_on_suite(
    model: ActorCritic,
    suite: Dict[str, Dict],
    loaded_models: Dict[str, ActorCritic],
    model_name: str,
    seed: int = 0,
    show_progress: bool = True,
    tqdm_position: int = 0,
    tqdm_leave: bool = False,
) -> Tuple[pd.DataFrame, Dict]:
    opponents_cfg = suite_spec_to_eval_cfg(suite)

    suite_df = trusted_evaluate_suite(
        policy=model,
        opponents=opponents_cfg,
        device=device,
        lookahead=_TRUSTED_LOOKAHEAD,
        seed=int(seed),
        swap_sides=True,
        policy_deterministic=MODEL_DETERMINISTIC,
        policy_temperature=1.0,
        progress=show_progress,
    )

    row = OrderedDict()
    row["MODEL"] = model_name

    details = {}
    for _, r in suite_df.iterrows():
        row_label = eval_label_to_row_label(str(r["opponent"]))
        row[row_label] = float(r["win_rate"])
        details[row_label] = {
            "wins": int(r.get("wins", 0)),
            "losses": int(r.get("losses", 0)),
            "draws": int(r.get("draws", 0)),
            "games": int(r.get("games", 0)),
            "win_rate": float(r.get("win_rate", 0.0)),
            "loss_rate": float(r.get("loss_rate", 0.0)),
            "draw_rate": float(r.get("draw_rate", 0.0)),
            "score": float(r.get("score", 0.0)),
            "avg_plies": float(r.get("avg_plies", 0.0)),
        }

    score_cols = [eval_label_to_row_label(str(x)) for x in suite_df["opponent"].tolist()]
    row["AVG_SCORE"] = float(np.mean([row[c] for c in score_cols])) if score_cols else 0.0

    hard_cols = [c for c in score_cols if c in {"LA-2", "LA-3", "LA-4", "LA-5", "LA-6", "LA-9", "LA-11", "LA-13"}]
    row["LA_HARD"] = float(np.mean([row[c] for c in hard_cols])) if hard_cols else 0.0

    global_score = float(trusted_global_score_from_suite_df(suite_df, base=float(GLOBAL_SCORE_WEIGHT_BASE)))
    row["GLOBAL_SCORE"] = global_score
    row["GS_CUSTOM"] = global_score  # backward-compatible alias

    df = pd.DataFrame([row])
    return df, details

print("Trusted evaluator override active: PPO.ppo_agent_eval.evaluate_suite + GLOBAL_SCORE")


Trusted evaluator override active: PPO.ppo_agent_eval.evaluate_suite + GLOBAL_SCORE



## Job selection and model loading

This cell:

- selects the active patch job(s)
- loads the required models
- shows which checkpoint is student / peer / adjudicator


In [5]:

if RUN_ALL_JOBS:
    JOBS_TO_RUN = PATCH_JOBS
else:
    JOBS_TO_RUN = [PATCH_JOBS[SELECT_JOB_INDEX]]

print("Jobs to run:")
for j_idx, job in enumerate(JOBS_TO_RUN):
    print(f"  [{j_idx}] {job['tag']}")

required_paths = set()
for job in JOBS_TO_RUN:
    required_paths.add(job["student_init"])
    required_paths.add(job["peer_teacher"])
    for p in job.get("adjudicator_models", []):
        required_paths.add(p)

loaded_models = {}
print("\nLoading required models...")
for path in sorted(required_paths):
    loaded_models[path] = load_actor_critic_from_ckpt(path, device=device, train=False)
    loaded_models[path].eval()
    for param in loaded_models[path].parameters():
        param.requires_grad_(False)
    print("Loaded:", path, "| prior weight:", f"{teacher_prior_weight(path):.3f}")


Jobs to run:
  [0] PPO_827_PATCH_FROM_832_LA2

Loading required models...
Loaded: PPO_Models/PPO_827.pt | prior weight: 1.355
Loaded: PPO_Models/PPO_832.pt | prior weight: 1.352
Loaded: PPO_Models/PPO_849.pt | prior weight: 1.470
Loaded: PPO_Models/X_1.pt | prior weight: 1.385



## Collection helpers

We collect:

- a **focus bank** from the student's weak matchup
- a **preserve bank** from matchups that should stay strong

Collection is student-centric on purpose. We want to patch what the student actually sees.


In [6]:


def sample_opening_prefix(seed: int, forced_seq: Optional[List[int]] = None) -> List[int]:
    rng = np.random.default_rng(seed)
    seq = list(forced_seq) if forced_seq else []

    if OPENING_MODE == "empty_only":
        return seq

    if OPENING_MODE == "forced_only":
        return seq

    if OPENING_MODE == "random_only":
        extra = int(rng.choice(EXTRA_RANDOM_OPENING_PLIES_CHOICES))
        for _ in range(extra):
            seq.append(int(rng.choice(CENTER_ORDER)))
        return seq

    if OPENING_MODE == "mixed":
        use_random = bool(rng.random() < OPENING_RANDOM_PROB)
        if use_random:
            extra = int(rng.choice(EXTRA_RANDOM_OPENING_PLIES_CHOICES))
            for _ in range(extra):
                seq.append(int(rng.choice(CENTER_ORDER)))
        return seq

    raise ValueError(f"Unknown OPENING_MODE: {OPENING_MODE}")

def play_match_collect_states(

    student_source,
    opponent_source,
    n_games: int,
    plan_idx: int,
    seed: int,
    max_positions_left: int,
    dedup_set: Optional[set],
    focus_mode: bool,
):
    rows = []
    env = make_env()

    for g in range(n_games):
        if len(rows) >= max_positions_left:
            break

        state = env.reset()
        ply_idx = 0
        student_starts = bool(g % 2 == 0)
        turn_is_student = student_starts
        done = False

        forced_prefix = sample_opening_prefix(seed + plan_idx * 1000 + g)

        for forced_move in forced_prefix:
            legal = env.available_actions()
            if forced_move not in legal:
                forced_move = center_tiebreak(legal)
            state, reward, done = env.step(forced_move)
            ply_idx += 1
            if done:
                break
            turn_is_student = not turn_is_student

        if done:
            continue

        result_final = None
        game_states = []

        for _ in range(42 - ply_idx):
            legal = env.available_actions()
            if not legal:
                result_final = 0
                break

            state_arr = np.asarray(state, dtype=np.float32)
            keep_this = (MIN_FOCUS_PLY <= ply_idx <= MAX_FOCUS_PLY) if focus_mode else (MIN_PRESERVE_PLY <= ply_idx <= MAX_PRESERVE_PLY)

            if keep_this:
                key = state_key(state_arr)
                if dedup_set is not None and key in dedup_set:
                    keep_this = False
                if keep_this:
                    game_states.append({
                        "state": state_arr.astype(np.float32, copy=False),
                        "legal_actions": np.array(legal, dtype=np.int8),
                        "source_plan_idx": int(plan_idx),
                        "source_game_idx": int(g),
                        "source_A": student_source["label"],
                        "source_B": opponent_source["label"],
                        "side_to_move_source": student_source["label"] if turn_is_student else opponent_source["label"],
                        "starter_source": student_source["label"] if student_starts else opponent_source["label"],
                        "ply": int(ply_idx),
                        "opening_prefix_plies": int(len(forced_prefix)),
                        "forced_opening_seq": ",".join(str(x) for x in forced_prefix),
                        "focus_source_game": bool(focus_mode),
                        "preserve_source_game": bool(not focus_mode),
                        "_dedup_key": key,
                    })

            chooser = student_source["choose"] if turn_is_student else opponent_source["choose"]
            action = int(chooser(state, legal))
            prev_turn_is_student = turn_is_student

            state, reward, done = env.step(action)
            ply_idx += 1

            if done:
                if reward > 0:
                    result_final = +1 if prev_turn_is_student else -1
                else:
                    result_final = 0
                break

            turn_is_student = not turn_is_student

        if result_final is None:
            result_final = 0

        if focus_mode and FOCUS_KEEP_ONLY_NONWINS and result_final > 0:
            continue
        if (not focus_mode) and PRESERVE_KEEP_ONLY_NONWINS and result_final > 0:
            continue

        for row in game_states:
            if len(rows) >= max_positions_left:
                break
            if dedup_set is not None:
                dedup_set.add(row.pop("_dedup_key"))
            else:
                row.pop("_dedup_key", None)
            row["student_result"] = int(result_final)
            rows.append(row)

    return rows

def collect_patch_states_for_job(job: Dict[str, Any], loaded_models: Dict[str, ActorCritic]) -> pd.DataFrame:
    all_rows = []
    dedup_focus_set = set() if DEDUP_FOCUS_STATES else None
    dedup_preserve_set = set() if DEDUP_PRESERVE_STATES else None
    student_source = make_source_from_spec({"type": "model", "path": job["student_init"]}, loaded_models=loaded_models, seed=SEED + 11)
    peer_source = make_source_from_spec({"type": "model", "path": job["peer_teacher"]}, loaded_models=loaded_models, seed=SEED + 13)

    plans = []

    focus_depth = int(job["focus_depth"])
    plans.append(("focus", {"type": "lookahead", "depth": focus_depth, "games": int(job["focus_games_student_vs_la"])}, "student_vs_focus"))
    plans.append(("focus", {"type": "lookahead", "depth": focus_depth, "games": int(job["focus_games_teacher_vs_la"])}, "peer_vs_focus"))

    for d in job["preserve_depths"]:
        plans.append(("preserve", {"type": "lookahead", "depth": int(d), "games": int(job["preserve_games_each"][int(d)])}, f"student_vs_LA{d}"))

    outer = tqdm(list(enumerate(plans, start=1)), desc="Collecting patch states", position=0, leave=True)

    focus_count = 0
    preserve_count = 0

    for plan_idx, (mode, opp_spec, label) in outer:
        focus_mode = (mode == "focus")
        max_left = (MAX_FOCUS_POSITIONS - focus_count) if focus_mode else (MAX_PRESERVE_POSITIONS - preserve_count)
        if max_left <= 0:
            continue

        opp_source = make_source_from_spec(opp_spec, loaded_models=loaded_models, seed=SEED + plan_idx * 100 + 7)
        player_source = peer_source if label == "peer_vs_focus" else student_source

        outer.set_postfix({
            "plan": plan_idx,
            "mode": mode,
            "player": player_source["label"],
            "opp": opp_source["label"],
            "rows": len(all_rows),
        })

        rows = play_match_collect_states(
            student_source=player_source,
            opponent_source=opp_source,
            n_games=int(opp_spec["games"]),
            plan_idx=plan_idx,
            seed=SEED + plan_idx * 1000,
            max_positions_left=max_left,
            dedup_set=(dedup_focus_set if focus_mode else dedup_preserve_set),
            focus_mode=focus_mode,
        )

        for r in rows:
            r["collection_label"] = label

        all_rows.extend(rows)

        if focus_mode:
            focus_count += len(rows)
        else:
            preserve_count += len(rows)

    return pd.DataFrame(all_rows)



## Target construction

For each row:

- **focus rows** try to repair the weakness
- **preserve rows** mostly stay close to the original student

On focus rows, the target is built from:

- peer teacher
- optional adjudicator checkpoints
- optional lookahead judge

This is intentionally much narrower than the old Track 2 routing.


In [7]:

def lookahead_judge_details(state: np.ndarray, legal_actions: List[int], depth: int) -> Dict[str, Any]:
    la = Connect4Lookahead()
    la.OPENING_RANDOM = False
    board = state_to_board_pov(state)
    scores = np.asarray(la.n_step_action_scores(board, player=1, depth=int(depth)), dtype=np.float64)
    mask = np.zeros(7, dtype=bool)
    mask[legal_actions] = True
    scores[~mask] = -1e18

    best = np.max(scores[mask])
    best_cols = [c for c in legal_actions if abs(scores[c] - best) <= 1e-12]
    top_action = center_tiebreak(best_cols)

    probs = np.zeros(7, dtype=np.float64)
    legal_scores = scores[mask]
    legal_scores = legal_scores - np.max(legal_scores)
    ex = np.exp(legal_scores)
    ex = ex / ex.sum()
    probs[mask] = ex

    return {
        "top_action": int(top_action),
        "probs": probs.astype(np.float32),
        "scores": scores.astype(np.float32),
    }

def build_patch_target_for_row(
    row: pd.Series,
    anchor_model: ActorCritic,
    peer_teacher_model: ActorCritic,
    adjudicator_models: List[ActorCritic],
    adjudicator_model_paths: List[str],
    la_judge_depth: Optional[int],
    focus_depth: int,
) -> Dict[str, Any]:
    state = row["state"]
    legal_actions = list(map(int, row["legal_actions"]))
    focus_row = bool(row["focus_source_game"])

    anchor_det = model_action_details(anchor_model, state, legal_actions)

    if not focus_row:
        probs = sharpen_distribution(anchor_det["probs"], power=SOFT_TARGET_POWER)
        return {
            "route_type": "preserve_anchor",
            "soft_target": probs.astype(np.float32),
            "hard_action": int(anchor_det["top_action"]),
            "target_value": float(anchor_det["value"]),
            "sample_weight": float(PRESERVE_WEIGHT),
            "anchor_weight": float(ANCHOR_LOSS_WEIGHT * ANCHOR_PRESERVE_MULT),
            "teacher_used": pretty_name("anchor"),
            "judge_action": None,
            "judge_disagreed": False,
        }

    peer_det = model_action_details(peer_teacher_model, state, legal_actions)
    peer_probs = np.asarray(peer_det["probs"], dtype=np.float64)
    weighted_sum = PEER_WEIGHT * peer_probs
    total_w = float(PEER_WEIGHT)

    teacher_used = [pretty_name("peer")]
    support_count = 1
    hard_action_votes = Counter([int(peer_det["top_action"])])

    for model_path, model in zip(adjudicator_model_paths, adjudicator_models):
        det = model_action_details(model, state, legal_actions)
        w = ADJUDICATOR_MODEL_WEIGHT * teacher_prior_weight(model_path)
        weighted_sum += w * np.asarray(det["probs"], dtype=np.float64)
        total_w += float(w)
        teacher_used.append(pretty_name(model_path))
        hard_action_votes[int(det["top_action"])] += 1
        if int(det["top_action"]) == int(peer_det["top_action"]):
            support_count += 1

    judge_action = None
    judge_disagreed = False
    if la_judge_depth is not None and la_judge_depth > 0:
        judge_det = lookahead_judge_details(state, legal_actions, depth=int(la_judge_depth))
        judge_probs = np.asarray(judge_det["probs"], dtype=np.float64)
        judge_action = int(judge_det["top_action"])
        judge_disagreed = (judge_action != int(peer_det["top_action"]))

        if judge_disagreed and USE_LA_JUDGE_HARD_TARGET_ON_FOCUS:
            blended = (1.0 - LA_JUDGE_BLEND) * (weighted_sum / max(total_w, 1e-12)) + LA_JUDGE_BLEND * judge_probs
            weighted_sum = blended
            total_w = 1.0
        else:
            weighted_sum += LA_JUDGE_WEIGHT * judge_probs
            total_w += float(LA_JUDGE_WEIGHT)

        if judge_action == int(peer_det["top_action"]):
            support_count += 1
        hard_action_votes[judge_action] += 1

    if REQUIRE_TEACHER_OR_ADJUDICATOR_SUPPORT and support_count < MIN_SUPPORT_COUNT_FOR_OVERRIDE:
        probs = sharpen_distribution(anchor_det["probs"], power=SOFT_TARGET_POWER)
        return {
            "route_type": "focus_fallback_anchor",
            "soft_target": probs.astype(np.float32),
            "hard_action": int(anchor_det["top_action"]),
            "target_value": float(anchor_det["value"]),
            "sample_weight": float(FOCUS_WEIGHT),
            "anchor_weight": float(ANCHOR_LOSS_WEIGHT * ANCHOR_FOCUS_MULT),
            "teacher_used": pretty_name("anchor"),
            "judge_action": judge_action,
            "judge_disagreed": bool(judge_disagreed),
        }

    blended_probs = sharpen_distribution(weighted_sum / max(total_w, 1e-12), power=SOFT_TARGET_POWER)

    if judge_action is not None and judge_disagreed:
        hard_action = judge_action
        route_type = "focus_judged"
    else:
        max_votes = max(hard_action_votes.values())
        vote_actions = [a for a, v in hard_action_votes.items() if v == max_votes]
        hard_action = center_tiebreak(vote_actions)
        route_type = "focus_peer"

    return {
        "route_type": route_type,
        "soft_target": blended_probs.astype(np.float32),
        "hard_action": int(hard_action),
        "target_value": float(peer_det["value"]),
        "sample_weight": float(FOCUS_WEIGHT),
        "anchor_weight": float(ANCHOR_LOSS_WEIGHT * ANCHOR_FOCUS_MULT),
        "teacher_used": ",".join(teacher_used),
        "judge_action": judge_action,
        "judge_disagreed": bool(judge_disagreed),
    }

def build_patch_dataset_for_job(job: Dict[str, Any], raw_df: pd.DataFrame, loaded_models: Dict[str, ActorCritic]):
    student_model = loaded_models[job["student_init"]]
    peer_teacher_model = loaded_models[job["peer_teacher"]]
    adjudicator_models = [loaded_models[p] for p in job.get("adjudicator_models", [])]
    adjudicator_model_paths = list(job.get("adjudicator_models", []))
    la_judge_depth = job.get("la_judge_depth", None)
    focus_depth = int(job["focus_depth"])

    records = []
    route_rows = []

    iterable = tqdm(raw_df.iterrows(), total=len(raw_df), desc="Routing patch rows", position=0, leave=True)
    for idx, (_, row) in enumerate(iterable):
        target = build_patch_target_for_row(
            row=row,
            anchor_model=student_model,
            peer_teacher_model=peer_teacher_model,
            adjudicator_models=adjudicator_models,
            adjudicator_model_paths=adjudicator_model_paths,
            la_judge_depth=la_judge_depth,
            focus_depth=focus_depth,
        )

        records.append({
            "state": row["state"],
            "soft_target": target["soft_target"],
            "hard_action": target["hard_action"],
            "target_value": target["target_value"],
            "sample_weight": target["sample_weight"],
            "anchor_weight": target["anchor_weight"],
            "focus_flag": bool(row["focus_source_game"]),
            "group_id": f"{row['source_plan_idx']}::{row['source_game_idx']}",
            "route_type": target["route_type"],
            "teacher_used": target["teacher_used"],
        })

        route_rows.append({
            "idx": idx,
            "route_type": target["route_type"],
            "hard_action": target["hard_action"],
            "teacher_used": target["teacher_used"],
            "judge_action": target["judge_action"],
            "judge_disagreed": target["judge_disagreed"],
            "source_A": row["source_A"],
            "source_B": row["source_B"],
            "source_game_idx": row["source_game_idx"],
            "side_to_move_source": row["side_to_move_source"],
            "ply": row["ply"],
            "opening_prefix_plies": row["opening_prefix_plies"],
            "forced_opening_seq": row["forced_opening_seq"],
            "focus_source_game": bool(row["focus_source_game"]),
            "preserve_source_game": bool(row["preserve_source_game"]),
            "student_result": int(row["student_result"]),
            "collection_label": row.get("collection_label", ""),
        })

        iterable.set_postfix({
            "route": target["route_type"],
            "records": len(records),
        })

    dataset = {
        "states": np.asarray([r["state"] for r in records], dtype=np.float32),
        "soft_targets": np.asarray([r["soft_target"] for r in records], dtype=np.float32),
        "hard_actions": np.asarray([r["hard_action"] for r in records], dtype=np.int64),
        "target_values": np.asarray([r["target_value"] for r in records], dtype=np.float32),
        "sample_weights": np.asarray([r["sample_weight"] for r in records], dtype=np.float32),
        "anchor_weights": np.asarray([r["anchor_weight"] for r in records], dtype=np.float32),
        "focus_flags": np.asarray([r["focus_flag"] for r in records], dtype=np.bool_),
        "group_ids": np.asarray([r["group_id"] for r in records], dtype=object),
        "route_types": np.asarray([r["route_type"] for r in records], dtype=object),
        "teacher_used": np.asarray([r["teacher_used"] for r in records], dtype=object),
    }
    route_df = pd.DataFrame(route_rows)
    return dataset, route_df



## Split and dataloaders

This uses a **group-aware split** by collected game, not a random row split.

That avoids almost-duplicate states leaking from train into valid.


In [8]:

def group_train_valid_split(dataset: Dict[str, Any], valid_frac: float = VALID_FRAC, seed: int = 0):
    rng = np.random.default_rng(seed)
    groups = np.asarray(dataset["group_ids"], dtype=object)
    route_types = np.asarray(dataset["route_types"], dtype=object)

    unique_groups = np.unique(groups)

    focus_groups = []
    preserve_groups = []
    for g in unique_groups:
        mask = (groups == g)
        is_focus = bool(np.any(route_types[mask] != "preserve_anchor"))
        if is_focus:
            focus_groups.append(g)
        else:
            preserve_groups.append(g)

    rng.shuffle(focus_groups)
    rng.shuffle(preserve_groups)

    n_valid_total = max(MIN_VALID_GROUPS, int(round(valid_frac * len(unique_groups))))
    n_valid_focus = min(len(focus_groups), max(MIN_FOCUS_VALID_GROUPS, int(round(valid_frac * len(focus_groups))))) if len(focus_groups) else 0
    n_valid_preserve = min(len(preserve_groups), max(MIN_PRESERVE_VALID_GROUPS, int(round(valid_frac * len(preserve_groups))))) if len(preserve_groups) else 0

    valid_groups = set(focus_groups[:n_valid_focus]) | set(preserve_groups[:n_valid_preserve])

    # top up if needed
    remaining = [g for g in unique_groups if g not in valid_groups]
    rng.shuffle(remaining)
    while len(valid_groups) < min(n_valid_total, len(unique_groups)) and remaining:
        valid_groups.add(remaining.pop())

    train_mask = np.array([g not in valid_groups for g in groups], dtype=bool)
    valid_mask = ~train_mask

    split = {
        "train_states": dataset["states"][train_mask],
        "train_soft": dataset["soft_targets"][train_mask],
        "train_hard": dataset["hard_actions"][train_mask],
        "train_value": dataset["target_values"][train_mask],
        "train_weight": dataset["sample_weights"][train_mask],
        "train_anchor_weight": dataset["anchor_weights"][train_mask],
        "train_route_types": dataset["route_types"][train_mask],
        "valid_states": dataset["states"][valid_mask],
        "valid_soft": dataset["soft_targets"][valid_mask],
        "valid_hard": dataset["hard_actions"][valid_mask],
        "valid_value": dataset["target_values"][valid_mask],
        "valid_weight": dataset["sample_weights"][valid_mask],
        "valid_anchor_weight": dataset["anchor_weights"][valid_mask],
        "valid_route_types": dataset["route_types"][valid_mask],
        "n_unique_groups_train": int(len(set(groups[train_mask]))),
        "n_unique_groups_valid": int(len(set(groups[valid_mask]))),
    }
    return split

def to_tensor(x, dtype, device=device):
    return torch.as_tensor(x, dtype=dtype, device=device)



## Training helpers

The student starts from the original checkpoint and is lightly patched.

Loss components:

- **soft target loss**
- **hard action loss**
- optional **value loss**
- **anchor KL** to preserve the original student policy

Best checkpoint selection uses a gameplay mini-eval, not only supervised loss.


In [9]:

def build_trainable_student(path: str) -> ActorCritic:
    student = load_actor_critic_from_ckpt(path, device=device, train=True)
    for p in student.parameters():
        p.requires_grad_(True)
    return student

@torch.no_grad()
def anchor_forward_probs(anchor_model: ActorCritic, states: torch.Tensor) -> torch.Tensor:
    logits, _ = anchor_model(states)
    return torch.softmax(logits, dim=-1)

def distill_batch(student: ActorCritic, anchor_model: ActorCritic, batch, optimizer=None, use_value_target=False):
    states, soft_target, hard_action, value_target, sample_weight, anchor_weight = batch
    logits, value_pred = student(states)
    probs = torch.softmax(logits, dim=-1)
    log_probs = torch.log_softmax(logits, dim=-1)

    soft_target = soft_target / soft_target.sum(dim=-1, keepdim=True).clamp_min(1e-12)

    loss_soft = -(soft_target * log_probs).sum(dim=-1)
    loss_hard = F.nll_loss(log_probs, hard_action, reduction="none")

    if use_value_target:
        loss_value = (value_pred.squeeze(-1) - value_target) ** 2
    else:
        loss_value = torch.zeros_like(loss_hard)

    with torch.no_grad():
        anchor_probs = anchor_forward_probs(anchor_model, states)
    loss_anchor = (anchor_probs * (anchor_probs.clamp_min(1e-12).log() - log_probs)).sum(dim=-1)

    total = (
        LOSS_W_SOFT * loss_soft
        + LOSS_W_HARD * loss_hard
        + LOSS_W_VALUE * loss_value
        + anchor_weight * loss_anchor
    )
    total = total * sample_weight
    loss = total.mean()

    if optimizer is not None:
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), MAX_GRAD_NORM)
        optimizer.step()

    with torch.no_grad():
        pred = torch.argmax(logits, dim=-1)
        hard_acc = (pred == hard_action).float().mean().item()
        entropy = (-(probs * log_probs).sum(dim=-1)).mean().item()

    return {
        "loss_total": float(loss.item()),
        "loss_soft": float(loss_soft.mean().item()),
        "loss_hard": float(loss_hard.mean().item()),
        "loss_value": float(loss_value.mean().item()),
        "loss_anchor": float(loss_anchor.mean().item()),
        "hard_acc": float(hard_acc),
        "entropy": float(entropy),
    }

@torch.no_grad()
def evaluate_supervised(student: ActorCritic, anchor_model: ActorCritic, valid_loader, use_value_target=False):
    student.eval()
    batch_metrics = []
    for batch in valid_loader:
        m = distill_batch(student, anchor_model, batch, optimizer=None, use_value_target=use_value_target)
        batch_metrics.append(m)
    student.train()

    return {
        "valid_total": float(np.mean([m["loss_total"] for m in batch_metrics])),
        "valid_soft": float(np.mean([m["loss_soft"] for m in batch_metrics])),
        "valid_hard": float(np.mean([m["loss_hard"] for m in batch_metrics])),
        "valid_value": float(np.mean([m["loss_value"] for m in batch_metrics])),
        "valid_anchor": float(np.mean([m["loss_anchor"] for m in batch_metrics])),
        "valid_hard_acc": float(np.mean([m["hard_acc"] for m in batch_metrics])),
        "valid_entropy": float(np.mean([m["entropy"] for m in batch_metrics])),
    }

def run_patch_training(
    job: Dict[str, Any],
    train_loader,
    valid_loader,
    loaded_models: Dict[str, ActorCritic],
):
    student = build_trainable_student(job["student_init"])
    anchor_model = loaded_models[job["student_init"]]
    focus_depth = int(job["focus_depth"])

    optimizer = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    history = []
    best_state = None
    best_epoch = None
    best_score = None
    patience_left = EARLY_STOPPING_PATIENCE

    print("Starting pairwise patch training...")
    print("Student init:", pretty_name(job["student_init"]))
    print("Peer teacher:", pretty_name(job["peer_teacher"]))
    print("Focus depth:", focus_depth)
    print("Adjudicator models:", [pretty_name(p) for p in job.get("adjudicator_models", [])])
    print("LA judge depth:", job.get("la_judge_depth", None))

    for epoch in range(1, EPOCHS + 1):
        student.train()
        batch_metrics = []

        for batch in train_loader:
            m = distill_batch(student, anchor_model, batch, optimizer=optimizer, use_value_target=False)
            batch_metrics.append(m)

        train_summary = {
            f"train_{k.replace('loss_', '')}" if k.startswith("loss_") else f"train_{k}": float(np.mean([m[k] for m in batch_metrics]))
            for k in batch_metrics[0].keys()
        }
        valid_summary = evaluate_supervised(student, anchor_model, valid_loader, use_value_target=False)

        epoch_row = {"epoch": epoch, "lr": float(optimizer.param_groups[0]["lr"])}
        epoch_row.update(train_summary)
        epoch_row.update(valid_summary)

        mini_df, _ = evaluate_model_on_suite(
            model=student,
            suite=MINI_EVAL_OPPONENTS,
            loaded_models=loaded_models,
            model_name=f"mini_epoch_{epoch}",
            seed=SEED + epoch,
            show_progress=True,
            tqdm_position=1,
            tqdm_leave=False,
        )
        epoch_row["mini_eval_GLOBAL_SCORE"] = float(mini_df.iloc[0]["GLOBAL_SCORE"])
        epoch_row["mini_eval_LA_HARD"] = float(mini_df.iloc[0]["LA_HARD"])
        epoch_row["mini_eval_AVG_SCORE"] = float(mini_df.iloc[0]["AVG_SCORE"])
        epoch_row["mini_eval_PATCH_SCORE"] = float(compute_patch_score_from_row(mini_df.iloc[0].to_dict(), focus_depth=focus_depth))

        history.append(epoch_row)

        current_score = float(epoch_row["mini_eval_PATCH_SCORE"])
        better = (best_score is None) or (current_score > best_score)

        if better:
            best_score = current_score
            best_epoch = epoch
            best_state = copy.deepcopy(student.net.state_dict())
            patience_left = EARLY_STOPPING_PATIENCE
            flag = "*"
        else:
            patience_left -= 1
            flag = ""

        print(
            f"Epoch {epoch:02d} | "
            f"train_total={epoch_row['train_total']:.4f} | "
            f"valid_total={epoch_row['valid_total']:.4f} | "
            f"valid_hard_acc={epoch_row['valid_hard_acc']:.4f} | "
            f"mini_GS={epoch_row['mini_eval_GLOBAL_SCORE']:.4f} | "
            f"mini_PATCH={epoch_row['mini_eval_PATCH_SCORE']:.4f} | "
            f"patience_left={patience_left} {flag}"
        )

        if patience_left <= 0:
            print("Early stopping triggered.")
            break

    if best_state is not None:
        student.net.load_state_dict(best_state, strict=True)
        print(f"Restored best student state from epoch {best_epoch}.")

    info = {
        "best_epoch": best_epoch,
        "best_score": best_score,
        "keep_best_by": KEEP_BEST_BY,
    }
    return student, pd.DataFrame(history), info



## Run jobs

This is the main execution section.

For each selected job, the notebook will:

1. evaluate the original student
2. collect patch states
3. build targets
4. split into train/valid
5. run patch training
6. evaluate the patched checkpoint
7. save artifacts


In [10]:

all_job_summaries = []

for job_idx, job in enumerate(JOBS_TO_RUN, start=1):
    print("\n" + "=" * 100)
    print(f"RUNNING JOB {job_idx}/{len(JOBS_TO_RUN)}: {job['tag']}")
    print("=" * 100)

    print("\nPair sanity check on EMPTY-BOARD eval suite (GLOBAL_SCORE-based):")
    display_pair_sanity(job=job, loaded_models=loaded_models, suite=DISTILL_EVAL_OPPONENTS, seed=SEED)

    print("\nEvaluating student init before patch...")
    pre_eval_df, pre_eval_details = evaluate_checkpoint_path_on_suite_cached(
        model_path=job["student_init"],
        suite=DISTILL_EVAL_OPPONENTS,
        loaded_models=loaded_models,
        seed=SEED,
        show_progress=True,
        tqdm_position=0,
        tqdm_leave=False,
        force_recompute=False,
    )
    display_eval(pre_eval_df)

    print("\nCollecting raw patch state pool...")
    raw_states_df = collect_patch_states_for_job(job, loaded_models=loaded_models)
    print("\nRaw states collected:", len(raw_states_df))
    display(raw_states_df.head())

    if len(raw_states_df) == 0:
        print("No states collected for this job, skipping.")
        continue

    print("\nRaw state quick diagnostics:")
    display(raw_states_df["ply"].describe().to_frame("ply"))
    display(raw_states_df.groupby(["source_A", "source_B"]).size().reset_index(name="count").sort_values("count", ascending=False).head(14))

    print("\nRouting patch rows...")
    patch_dataset, route_df = build_patch_dataset_for_job(job, raw_states_df, loaded_models=loaded_models)

    print("\nPatch dataset size:", len(patch_dataset["hard_actions"]))
    display(route_df.head())

    print("\nRoute type counts:")
    route_type_counts_df = route_df["route_type"].value_counts(dropna=False).rename_axis("route_type").reset_index(name="count")
    display(route_type_counts_df)

    print("\nTeacher used counts:")
    teacher_used_counts_df = route_df["teacher_used"].value_counts(dropna=False).rename_axis("teacher_used").reset_index(name="count")
    display(teacher_used_counts_df.head(20))

    print("\nJudge disagreement counts:")
    judge_dis_counts_df = route_df["judge_disagreed"].value_counts(dropna=False).rename_axis("judge_disagreed").reset_index(name="count")
    display(judge_dis_counts_df)

    split = group_train_valid_split(patch_dataset, valid_frac=VALID_FRAC, seed=SEED)
    print("\nTrain size:", len(split["train_states"]))
    print("Valid size:", len(split["valid_states"]))
    print("Unique train groups:", split["n_unique_groups_train"])
    print("Unique valid groups:", split["n_unique_groups_valid"])

    train_route_counts = pd.Series(split["train_route_types"]).value_counts(dropna=False).rename_axis("route_type").reset_index(name="train_count")
    valid_route_counts = pd.Series(split["valid_route_types"]).value_counts(dropna=False).rename_axis("route_type").reset_index(name="valid_count")
    display(train_route_counts)
    display(valid_route_counts)

    train_ds = TensorDataset(
        to_tensor(split["train_states"], torch.float32),
        to_tensor(split["train_soft"], torch.float32),
        to_tensor(split["train_hard"], torch.long),
        to_tensor(split["train_value"], torch.float32),
        to_tensor(split["train_weight"], torch.float32),
        to_tensor(split["train_anchor_weight"], torch.float32),
    )
    valid_ds = TensorDataset(
        to_tensor(split["valid_states"], torch.float32),
        to_tensor(split["valid_soft"], torch.float32),
        to_tensor(split["valid_hard"], torch.long),
        to_tensor(split["valid_value"], torch.float32),
        to_tensor(split["valid_weight"], torch.float32),
        to_tensor(split["valid_anchor_weight"], torch.float32),
    )

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
    valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

    student_model, train_history_df, train_info = run_patch_training(
        job=job,
        train_loader=train_loader,
        valid_loader=valid_loader,
        loaded_models=loaded_models,
    )

    display(train_history_df)
    print("\nTraining info:", train_info)

    # Save checkpoint before post eval so repeated experiments can reuse cache.
    dataset_path = DATA_DIR / f"{job['output_tag']}_patch_dataset.pkl"
    route_summary_path = DATA_DIR / f"{job['output_tag']}_route_summary.xlsx"
    results_xlsx_path = OUTPUT_DIR / f"{job['output_tag']}_results.xlsx"
    model_path = MODEL_DIR / f"{job['output_tag']}.pt"
    json_summary_path = OUTPUT_DIR / f"{job['output_tag']}_summary.json"

    if SAVE_MODEL_CHECKPOINT:
        save_cnet192(
            model=student_model.net,
            path=model_path,
            tag=job["output_tag"],
            session=job["output_tag"],
            episode=0,
            seed=SEED,
            t=0,
            cfg_override={"use_mid_3x3": bool(getattr(student_model.net, "use_mid_3x3", True))},
        )
        print("Saved patched checkpoint to:", model_path)

        # refresh loaded model for cached eval
        loaded_models[str(model_path)] = load_actor_critic_from_ckpt(model_path, device=device, train=False)
        loaded_models[str(model_path)].eval()
        for param in loaded_models[str(model_path)].parameters():
            param.requires_grad_(False)

    print("\nEvaluating patched student...")
    post_eval_df, post_eval_details = evaluate_checkpoint_path_on_suite_cached(
        model_path=model_path if SAVE_MODEL_CHECKPOINT else job["student_init"],
        suite=DISTILL_EVAL_OPPONENTS,
        loaded_models=loaded_models,
        seed=SEED,
        show_progress=True,
        tqdm_position=0,
        tqdm_leave=False,
        force_recompute=not SAVE_MODEL_CHECKPOINT,
    )
    post_eval_df.loc[0, "MODEL"] = job["output_tag"]
    display_eval(post_eval_df)

    print("\nBefore / after comparison:")
    display_before_after(pre_eval_df, post_eval_df)

    if SAVE_PATCH_DATASET:
        with open(dataset_path, "wb") as f:
            pickle.dump({
                "job": job,
                "patch_dataset": patch_dataset,
            }, f)
        print("Saved patch dataset to:", dataset_path)

    if SAVE_RESULTS_XLSX:
        with pd.ExcelWriter(results_xlsx_path, engine="openpyxl") as writer:
            raw_states_df.to_excel(writer, sheet_name="raw_states", index=False)
            route_df.to_excel(writer, sheet_name="routes", index=False)
            route_type_counts_df.to_excel(writer, sheet_name="route_type_counts", index=False)
            teacher_used_counts_df.to_excel(writer, sheet_name="teacher_used_counts", index=False)
            judge_dis_counts_df.to_excel(writer, sheet_name="judge_dis_counts", index=False)
            train_history_df.to_excel(writer, sheet_name="train_history", index=False)
            pre_eval_df.to_excel(writer, sheet_name="pre_eval", index=False)
            post_eval_df.to_excel(writer, sheet_name="post_eval", index=False)
        print("Saved results workbook to:", results_xlsx_path)

    summary_payload = {
        "job": job,
        "training_config": {
            "BATCH_SIZE": BATCH_SIZE,
            "LR": LR,
            "EPOCHS": EPOCHS,
            "LOSS_W_SOFT": LOSS_W_SOFT,
            "LOSS_W_HARD": LOSS_W_HARD,
            "ANCHOR_LOSS_WEIGHT": ANCHOR_LOSS_WEIGHT,
            "ANCHOR_FOCUS_MULT": ANCHOR_FOCUS_MULT,
            "ANCHOR_PRESERVE_MULT": ANCHOR_PRESERVE_MULT,
            "KEEP_BEST_BY": KEEP_BEST_BY,
            "OPENING_MODE": OPENING_MODE,
            "VALID_FRAC": VALID_FRAC,
        },
        "dataset": {
            "n_raw_states": int(len(raw_states_df)),
            "n_patch_states": int(len(patch_dataset["hard_actions"])),
            "route_type_counts": route_type_counts_df.to_dict(orient="records"),
            "teacher_used_counts": teacher_used_counts_df.to_dict(orient="records"),
            "judge_disagreement_counts": judge_dis_counts_df.to_dict(orient="records"),
        },
        "training": {
            "best_epoch": int(train_info["best_epoch"]) if train_info["best_epoch"] is not None else None,
            "best_score": float(train_info["best_score"]) if train_info["best_score"] is not None else None,
            "history": train_history_df.to_dict(orient="records"),
        },
        "pre_eval": pre_eval_df.iloc[0].to_dict(),
        "post_eval": post_eval_df.iloc[0].to_dict(),
    }

    if SAVE_JSON_SUMMARY:
        json_summary_path.write_text(json.dumps(summary_payload, indent=2, default=str), encoding="utf-8")
        print("Saved JSON summary to:", json_summary_path)

    all_job_summaries.append({
        "job_tag": job["tag"],
        "output_tag": job["output_tag"],
        "student_init": pretty_name(job["student_init"]),
        "peer_teacher": pretty_name(job["peer_teacher"]),
        "focus_depth": int(job["focus_depth"]),
        "pre_GLOBAL_SCORE": float(pre_eval_df.iloc[0]["GLOBAL_SCORE"]),
        "post_GLOBAL_SCORE": float(post_eval_df.iloc[0]["GLOBAL_SCORE"]),
        "pre_LA_HARD": float(pre_eval_df.iloc[0]["LA_HARD"]),
        "post_LA_HARD": float(post_eval_df.iloc[0]["LA_HARD"]),
        "pre_AVG_SCORE": float(pre_eval_df.iloc[0]["AVG_SCORE"]),
        "post_AVG_SCORE": float(post_eval_df.iloc[0]["AVG_SCORE"]),
        "checkpoint": str(model_path),
    })

summary_df = pd.DataFrame(all_job_summaries)
if len(summary_df) > 0:
    print("\nJob summary:")
    display(summary_df)

if SAVE_EVAL_CACHE:
    save_eval_cache()
    print("\nSaved eval cache to:", EVAL_CACHE_PATH)



RUNNING JOB 1/1: PPO_827_PATCH_FROM_832_LA2

Pair sanity check on EMPTY-BOARD eval suite (GLOBAL_SCORE-based):
Sanity-check models:
   PPO_827 -> PPO_Models/PPO_827.pt
   PPO_832 -> PPO_Models/PPO_832.pt
   X_1 -> PPO_Models/X_1.pt
   PPO_849 -> PPO_Models/PPO_849.pt


Pair sanity:   0%|          | 0/4 [00:00<?, ?it/s]

----------------------------------------------------------------------------------------
Sanity step 1/4: PPO_827
[CACHE HIT] PPO_827 | seed=666
----------------------------------------------------------------------------------------
Sanity step 2/4: PPO_832
[CACHE HIT] PPO_832 | seed=666
----------------------------------------------------------------------------------------
Sanity step 3/4: X_1
[CACHE HIT] X_1 | seed=666
----------------------------------------------------------------------------------------
Sanity step 4/4: PPO_849
[CACHE HIT] PPO_849 | seed=666
Pair sanity table:


,MODEL,MODEL_PATH,LA-2,LA-3,LA-4,LA-5,LA-6,LA-9,LA-11,LA-13,AVG_SCORE,LA_HARD,GLOBAL_SCORE
1,PPO_832,PPO_Models/PPO_832.pt,1.0,0.5,1.0,0.5,1.0,1.0,1.0,1.0,0.923077,0.875,0.977046
2,X_1,PPO_Models/X_1.pt,1.0,0.5,1.0,1.0,0.5,1.0,1.0,1.0,0.923077,0.875,0.970966
3,PPO_849,PPO_Models/PPO_849.pt,1.0,0.5,1.0,1.0,0.5,1.0,1.0,1.0,0.923077,0.875,0.970966
0,PPO_827,PPO_Models/PPO_827.pt,0.5,1.0,1.0,1.0,0.5,1.0,1.0,1.0,0.922115,0.875,0.973111



Evaluating student init before patch...
[CACHE HIT] PPO_827 | seed=666


...

Metric [GLOBAL_SCORE] = 0.9502275404133707

Before / after comparison:


,STAGE,MODEL,LA-2,LA-3,LA-4,LA-5,LA-6,LA-9,LA-11,LA-13,AVG_SCORE,LA_HARD,GLOBAL_SCORE
0,before,PPO_827,0.5,1.0,1.0,1.0,0.5,1.0,1.0,1.0,0.922115,0.875,0.973111
1,after,PPO_827_P1776703120,0.5,0.5,1.0,0.5,0.5,1.0,1.0,1.0,0.846154,0.750,0.950228


Saved patch dataset to: PAIRWISE_PATCH\data\PPO_827_P1776703120_patch_dataset.pkl
Saved results workbook to: PAIRWISE_PATCH\PPO_827_P1776703120_results.xlsx
Saved JSON summary to: PAIRWISE_PATCH\PPO_827_P1776703120_summary.json

Job summary:


,job_tag,output_tag,student_init,peer_teacher,focus_depth,pre_GLOBAL_SCORE,post_GLOBAL_SCORE,pre_LA_HARD,post_LA_HARD,pre_AVG_SCORE,post_AVG_SCORE,checkpoint
0,PPO_827_PATCH_FROM_832_LA2,PPO_827_P1776703120,PPO_827,PPO_832,2,0.973111,0.950228,0.875,0.75,0.922115,0.846154,PPO_Models\PPO_827_P1776703120.pt



Saved eval cache to: PAIRWISE_PATCH\pairwise_eval_cache.json



## Notes

This notebook is deliberately narrow.

Use it when you have evidence like:

- model A is bad against `LA-x`
- model B is better against `LA-x`

and you want a controlled repair pass.

A few practical rules:

- keep the patch small
- preserve the student's existing strengths
- trust lookahead adjudication more than PPO politics when in doubt
- judge success by gameplay, not by lower supervised loss
